In [1]:
from pathlib import Path

import numpy as np
import torch
from IPython.lib.display import Audio
from sklearn.model_selection import train_test_split, StratifiedKFold
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter

from CustomSpeachDataset import CustomSpeachDataset
from SpeachClassifierModel import SpeachClassifierModel
from augmentations import audio_transform

In [2]:
device = device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [3]:
MODELS_DIR = Path("checkpoints")
MODELS_DIR.mkdir(exist_ok=True)

In [4]:
dataset = CustomSpeachDataset(Path("preprocessed_dataset"), preload=True, data_transforms=audio_transform)
dataset.to(device)

Preloading dataset from disk...


/home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
audio_transform

Compose(
    RandomApply(
    p=0.5
    Gain()
)
    RandomApply(
    p=0.2
    Noise()
)
    RandomApply(
    p=0.3
    FadeIn()
)
    RandomApply(
    p=0.3
    FadeOut()
)
)

In [6]:
train_val_indices, test_indices = train_test_split(
    range(len(dataset)),
    test_size=0.2,
    stratify=dataset.y.cpu(),
    random_state=42
)

class_weights = 1.0 / dataset.label_counts
all_sample_weights = np.array([class_weights[y] for y in dataset.y.cpu()])

y_train = dataset.y.cpu()[train_val_indices]
train_val_dataset = torch.utils.data.Subset(dataset, train_val_indices)
train_val_sample_weights = all_sample_weights[train_val_indices]
len(train_val_dataset), len(y_train)

(14679, 14679)

In [7]:
Audio(train_val_dataset[5][0].cpu(), rate=8000)

In [8]:
def train_new_model(writer: SummaryWriter, fold: int, params: dict, train_loader: DataLoader, val_loader: DataLoader, save_dir: Path, training_state: dict|None) -> SpeachClassifierModel:
    model = SpeachClassifierModel(params["dropout_rate"])
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])

    best_val_loss = 99999999999.9

    loss_fn = nn.CrossEntropyLoss()

    start_epoch = 0

    if training_state:
        optimizer.load_state_dict(training_state["optimizer"])
        best_val_loss = training_state["best_val_loss"]
        start_epoch = training_state["epoch"]

    for epoch in range(start_epoch, params["epochs"]):
        print(f"Fold {fold} starting epoch {epoch}")

        model.train()
        train_loss = 0.0
        for data, target in train_loader:
            optimizer.zero_grad()

            y_pred = model(data)

            loss = loss_fn(y_pred, target)
            train_loss += loss.item()

            loss.backward()
            optimizer.step()

        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for data, target in val_loader:
                y_pred = model(data)
                loss = loss_fn(y_pred, target)
                val_loss += loss.item()
            val_loss /= len(val_loader)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_path = save_dir / "best" / f"fold{fold}.pth"
            torch.save(model.state_dict(), save_path)

        writer.add_scalar(f"Loss/train/fold{fold}", train_loss, epoch)
        writer.add_scalar(f"Loss/val/fold{fold}", val_loss, epoch)

        torch.save(model.state_dict(), save_dir / "latest" / f"fold{fold}.pth")

        training_state = {
            "epoch": epoch,
            "optimizer": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
            "fold": fold,
        }
        torch.save(training_state, save_dir / "latest" / f"training_state.pth")

    model.best_val_loss = best_val_loss

    return model

In [9]:
def train_models_5cv(writer: SummaryWriter, params: dict, skf: StratifiedKFold, save_dir: Path, continue_training: bool = False):
    start_fold = 0

    if continue_training:
        training_state = torch.load(save_dir / "latest" / f"training_state.pth")
        start_fold = training_state["fold"]
    else:
        training_state = None

    for fold, (train_index, val_index) in enumerate(skf.split(y_train, y_train)):
        if fold < start_fold:
            print(f"fold {fold} skipped")
            continue

        print(f"starting training fold {fold}")

        train_sampler = WeightedRandomSampler(
            weights=train_val_sample_weights[train_index],
            num_samples=len(train_val_sample_weights[train_index]),
            replacement=True,
        )
        train_subset = torch.utils.data.Subset(train_val_dataset, train_index)
        train_loader = DataLoader(train_subset, batch_size=32, sampler=train_sampler)

        val_subset = torch.utils.data.Subset(train_val_dataset, val_index)
        val_loader = DataLoader(val_subset, batch_size=1024)

        train_new_model(writer, fold, params, train_loader, val_loader, save_dir, training_state)


In [10]:
params = {
    "lr": 0.001,
    "dropout_rate": 0.15,
    "weight_decay": 0.0006,
    "epochs": 100,
}
save_dir = MODELS_DIR / "training_augmented"
save_dir.mkdir(parents=True, exist_ok=True)

(save_dir/"best").mkdir(parents=True, exist_ok=True)
(save_dir/"latest").mkdir(parents=True, exist_ok=True)
torch.save(params, save_dir / "best" / "model_params.pth")
torch.save(params, save_dir / "latest" / "model_params.pth")

skf = StratifiedKFold(n_splits=5, shuffle=True)

with SummaryWriter(f"runs/augmented_train/folds") as writer:
    train_models_5cv(writer, params, skf, save_dir, continue_training=False)


starting training fold 0
Fold 0 starting epoch 0
Fold 0 starting epoch 1
Fold 0 starting epoch 2
Fold 0 starting epoch 3
Fold 0 starting epoch 4
Fold 0 starting epoch 5
Fold 0 starting epoch 6
Fold 0 starting epoch 7
Fold 0 starting epoch 8
Fold 0 starting epoch 9
Fold 0 starting epoch 10
Fold 0 starting epoch 11
Fold 0 starting epoch 12
Fold 0 starting epoch 13
Fold 0 starting epoch 14
Fold 0 starting epoch 15
Fold 0 starting epoch 16
Fold 0 starting epoch 17
Fold 0 starting epoch 18
Fold 0 starting epoch 19
Fold 0 starting epoch 20
Fold 0 starting epoch 21
Fold 0 starting epoch 22
Fold 0 starting epoch 23
Fold 0 starting epoch 24
Fold 0 starting epoch 25
Fold 0 starting epoch 26
Fold 0 starting epoch 27
Fold 0 starting epoch 28
Fold 0 starting epoch 29
Fold 0 starting epoch 30
Fold 0 starting epoch 31
Fold 0 starting epoch 32
Fold 0 starting epoch 33
Fold 0 starting epoch 34
Fold 0 starting epoch 35
Fold 0 starting epoch 36
Fold 0 starting epoch 37
Fold 0 starting epoch 38
Fold 0 sta